In [ ]:
# ============================================================
# Stance Detection using BiLSTM + GloVe Embeddings
# Ready to Run in Google Colab
# ============================================================

# =========================
# Install / Import Libraries
# =========================
!pip install nltk -q

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout,
    BatchNormalization
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

# =========================
# Download NLTK Stopwords
# =========================
nltk.download("stopwords")

STOPWORDS = set(stopwords.words("english"))

# =========================
# Upload Dataset in Colab
# =========================
from google.colab import files

print("Upload SemEval dataset CSV file")
uploaded = files.upload()

# Get uploaded filename
data_path = list(uploaded.keys())[0]

# =========================
# Load Dataset
# =========================
df = pd.read_csv(data_path)

print("Dataset Shape:", df.shape)
print(df.head())

# =========================
# Handle Missing Values
# =========================
df.dropna(subset=["Stance"], inplace=True)

# =========================
# Map Stance Labels
# =========================
stance_mapping = {
    "FAVOR": 0,
    "AGAINST": 1,
    "NONE": 2
}

df["label"] = df["Stance"].map(stance_mapping)

# Remove invalid labels
df.dropna(subset=["label"], inplace=True)

# Convert to integer
df["label"] = df["label"].astype(int)

print("\nLabel Distribution:")
print(df["label"].value_counts())

# =========================
# Text Preprocessing
# =========================
def preprocess_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Remove punctuation and numbers
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Remove stopwords
    text = " ".join([
        word for word in text.split()
        if word not in STOPWORDS
    ])

    return text

# Apply preprocessing
df["cleaned_tweet"] = df["Tweet"].astype(str).apply(preprocess_text)

print("\nSample Cleaned Tweets:")
print(df[["Tweet", "cleaned_tweet"]].head())

# =========================
# Tokenization and Padding
# =========================
vocab_size = 20000
max_length = 100
oov_token = "<OOV>"

tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token=oov_token
)

tokenizer.fit_on_texts(df["cleaned_tweet"])

# Convert text to sequences
sequences = tokenizer.texts_to_sequences(df["cleaned_tweet"])

# Pad sequences
padded_sequences = pad_sequences(
    sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

# =========================
# Train / Validation / Test Split
# =========================
X = padded_sequences
y = df["label"].values

# 70% Train | 30% Temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# 15% Validation | 15% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("\nTrain Shape:", X_train.shape)
print("Validation Shape:", X_val.shape)
print("Test Shape:", X_test.shape)

# =========================
# Upload GloVe File
# =========================
print("\nUpload glove.6B.100d.txt")
uploaded_glove = files.upload()

glove_path = list(uploaded_glove.keys())[0]

# =========================
# Load GloVe Embeddings
# =========================
embedding_dim = 100
embedding_index = {}

print("\nLoading GloVe embeddings...")

with open(glove_path, encoding="utf-8") as f:
    for line in f:
        values = line.split()

        word = values[0]
        coefs = np.asarray(values[1:], dtype="float32")

        embedding_index[word] = coefs

print("Total word vectors loaded:", len(embedding_index))

# =========================
# Create Embedding Matrix
# =========================
word_index = tokenizer.word_index

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in word_index.items():

    if i < vocab_size:

        embedding_vector = embedding_index.get(word)

        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

print("Embedding Matrix Shape:", embedding_matrix.shape)

# =========================
# Build BiLSTM Model
# =========================
model = Sequential([

    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_length,
        trainable=False
    ),

    Bidirectional(
        LSTM(
            128,
            return_sequences=True,
            dropout=0.3,
            recurrent_dropout=0.3
        )
    ),

    Bidirectional(
        LSTM(
            64,
            dropout=0.3,
            recurrent_dropout=0.3
        )
    ),

    BatchNormalization(),

    Dense(
        128,
        activation="relu",
        kernel_regularizer=l2(0.01)
    ),

    Dropout(0.5),

    Dense(3, activation="softmax")

])

# =========================
# Compile Model
# =========================
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Display Model Summary
model.summary()

# =========================
# Early Stopping
# =========================
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

# =========================
# Train Model
# =========================
epochs = 20
batch_size = 32

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stopping],
    verbose=1
)

# =========================
# Evaluate Model
# =========================
loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# =========================
# Predictions
# =========================
y_pred_probs = model.predict(X_test)

y_pred = np.argmax(y_pred_probs, axis=-1)

# =========================
# Classification Report
# =========================
print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["FAVOR", "AGAINST", "NONE"]
    )
)

# =========================
# Confusion Matrix
# =========================
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["FAVOR", "AGAINST", "NONE"],
    yticklabels=["FAVOR", "AGAINST", "NONE"]
)

plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix")

plt.show()

# =========================
# Accuracy and F1 Score
# =========================
acc = accuracy_score(y_test, y_pred)

f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

print(f"\nAccuracy Score: {acc:.4f}")
print(f"Weighted F1 Score: {f1:.4f}")

# =========================
# Plot Training History
# =========================
plt.figure(figsize=(12, 5))

# Accuracy Plot
plt.subplot(1, 2, 1)

plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

# Loss Plot
plt.subplot(1, 2, 2)

plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()

# =========================
# Save Model
# =========================
model.save("stance_classification_model.h5")

print("\nModel saved successfully!")
print("File Name: stance_classification_model.h5")